# GoPro 屏幕提取器
**用法：** 点击顶部菜单 **Runtime → Run all**，等待最后一格出现链接，点击即可使用。

无需注册任何账号。

In [ ]:
# 1. 安装依赖（约1分钟）
!pip install -q gradio opencv-python-headless Pillow

In [ ]:
# 2. 核心提取逻辑
import cv2
import numpy as np
import os, uuid, tempfile

def enhance_frame(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    frame = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    blur = cv2.GaussianBlur(frame, (0, 0), 3)
    return cv2.addWeighted(frame, 1.5, blur, -0.5, 0)

def order_points(pts):
    rect = np.zeros((4, 2), dtype='float32')
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

def detect_screen_contour(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    gray = cv2.bilateralFilter(gray, 9, 75, 75)
    edges = cv2.Canny(gray, 30, 100)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    edges = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    frame_area = h * w
    best, best_area = None, 0
    for cnt in contours[:15]:
        area = cv2.contourArea(cnt)
        if area < frame_area * 0.05 or area > frame_area * 0.95:
            continue
        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
        if len(approx) == 4:
            pts = approx.reshape(4, 2).astype('float32')
            ordered = order_points(pts)
            width = np.linalg.norm(ordered[1] - ordered[0])
            height = np.linalg.norm(ordered[3] - ordered[0])
            if height == 0: continue
            ratio = width / height
            if 0.5 < ratio < 3.0 and area > best_area:
                best, best_area = ordered, area
    return best

def compute_output_size(pts):
    tl, tr, br, bl = pts
    W = int(max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl)))
    H = int(max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl)))
    return (W // 2) * 2, (H // 2) * 2

def warp_frame(frame, pts, W, H):
    dst = np.array([[0,0],[W-1,0],[W-1,H-1],[0,H-1]], dtype='float32')
    M = cv2.getPerspectiveTransform(pts, dst)
    return cv2.warpPerspective(frame, M, (W, H))

def smooth_corners(history, new_pts, alpha=0.3):
    if not history: return new_pts
    return history[-1] * (1 - alpha) + new_pts * alpha

def refine_corners_with_flow(prev_gray, curr_gray, prev_pts):
    pts_flat = prev_pts.reshape(4, 1, 2).astype('float32')
    next_pts, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_gray, curr_gray, pts_flat, None,
        winSize=(31,31), maxLevel=4,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    if status is None or status.sum() < 4: return None
    return next_pts.reshape(4, 2)

def process_video(input_path, output_path, progress_gr=None):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise ValueError('Cannot open video')
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    initial_pts = None
    for _ in range(min(90, total_frames)):
        ret, frame = cap.read()
        if not ret: break
        pts = detect_screen_contour(frame)
        if pts is not None:
            initial_pts = pts
            break
    if initial_pts is None:
        cap.release()
        raise ValueError('未能检测到屏幕矩形，请确认 GoPro 屏幕在视频中清晰可见。')
    out_W, out_H = compute_output_size(initial_pts)
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (out_W, out_H))
    current_pts = initial_pts.copy()
    corner_history = [current_pts]
    prev_gray = None
    frame_idx = 0
    fail_count = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if prev_gray is not None:
            flow_pts = refine_corners_with_flow(prev_gray, gray, current_pts)
            if flow_pts is not None:
                fail_count = 0
                current_pts = smooth_corners(corner_history, flow_pts)
                corner_history.append(current_pts)
            else:
                fail_count += 1
        if prev_gray is None or fail_count >= 15:
            new_pts = detect_screen_contour(frame)
            if new_pts is not None:
                current_pts = smooth_corners(corner_history, new_pts, alpha=0.25)
                corner_history.append(current_pts)
                fail_count = 0
        prev_gray = gray
        try:
            writer.write(enhance_frame(warp_frame(frame, current_pts, out_W, out_H)))
        except Exception:
            writer.write(np.zeros((out_H, out_W, 3), dtype=np.uint8))
        frame_idx += 1
        if progress_gr and frame_idx % 15 == 0:
            pct = int(frame_idx / total_frames * 100)
            progress_gr(pct, desc=f'处理中 {pct}%  ({frame_idx}/{total_frames} 帧)')
    cap.release()
    writer.release()
    return output_path

print('处理逻辑加载完成 ✓')

In [ ]:
# 3. 启动 Gradio 界面（自动生成公开链接，无需注册）
import gradio as gr
import tempfile, os

def run_extraction(video_path, progress=gr.Progress()):
    if video_path is None:
        raise gr.Error('请先上传视频')
    progress(0, desc='开始处理...')
    out_path = tempfile.mktemp(suffix='_extracted.mp4')
    try:
        result = process_video(video_path, out_path, progress_gr=progress)
    except ValueError as e:
        raise gr.Error(str(e))
    progress(1.0, desc='完成！')
    return result

with gr.Blocks(title='GoPro 屏幕提取器', theme=gr.themes.Base()) as demo:
    gr.Markdown('# 🎥 GoPro 屏幕提取器\n上传包含 GoPro 屏幕的视频，自动锁定屏幕、矫正透视并增强画质。')
    with gr.Row():
        with gr.Column():
            video_in = gr.Video(label='上传视频（MP4 / MOV / AVI）', sources=['upload'])
            btn = gr.Button('开始提取', variant='primary')
        with gr.Column():
            video_out = gr.Video(label='提取结果', interactive=False)
    btn.click(fn=run_extraction, inputs=video_in, outputs=video_out)

demo.launch(share=True, debug=False)
